[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mubarekidriss/voice-cloner/blob/main/voice_cloner_demo.ipynb)

# Voice Cloner - ICS 499 Capstone (Group 12)

**Team:** Mubarek Idris & Mikita Misiulia

Type text + record/upload a 6-10s voice sample -> hear it in a **standard AI voice** (gTTS) and a **cloned voice** (Coqui XTTS v2), side by side.

**Setup:** Runtime -> Change runtime type -> **GPU (T4)**, then Runtime -> **Run all**. First run downloads ~2 GB of model, allow 3-5 minutes. Open the `gradio.live` link printed by the last cell.

**License:** running this notebook accepts the [Coqui Public Model License (CPML)](https://coqui.ai/cpml) for the XTTS v2 model. The `COQUI_TOS_AGREED` flag below is how Coqui records that acceptance - by running the cells, you accept the license.

**Responsible use:** only clone a voice with that person's permission.

In [ ]:
# Install dependencies (Colab already has PyTorch). transformers 5.2+ removed
# isin_mps_friendly which coqui-tts imports, so pin transformers below 5.
!pip install -q coqui-tts "transformers>=4.57,<5" gTTS gradio

In [ ]:
# Sanity check: a GPU makes XTTS usable. If this prints 'cpu', switch the runtime to GPU.
import torch
print('cuda' if torch.cuda.is_available() else 'cpu - WARNING: go to Runtime -> Change runtime type -> GPU')

In [ ]:
import os
# Running this notebook means YOU accept the Coqui Public Model License (see top cell).
# This flag is how Coqui records that acceptance so the model loads non-interactively.
os.environ['COQUI_TOS_AGREED'] = '1'

from TTS.api import TTS

MODEL_NAME = 'tts_models/multilingual/multi-dataset/xtts_v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading {MODEL_NAME} on {device} (first run downloads about 2 GB)...')
tts = TTS(MODEL_NAME).to(device)
print('Model loaded.')

In [ ]:
import tempfile
import gradio as gr
from gtts import gTTS

def clone_voices(text, reference_audio):
    """Generate (standard_voice_audio, cloned_voice_audio) for the given text."""
    if not text or not text.strip():
        raise gr.Error('Type some text first.')
    if reference_audio is None:
        raise gr.Error('Record or upload a voice sample first (6-10 seconds works best).')

    text = text.strip()

    # 1. Standard AI voice: gTTS always sounds the same, no sample needed.
    standard_path = tempfile.mktemp(suffix='_standard.mp3')
    gTTS(text=text, lang='en').save(standard_path)

    # 2. Cloned voice: XTTS v2 copies the voice in the reference sample.
    clone_path = tempfile.mktemp(suffix='_cloned.wav')
    tts.tts_to_file(text=text, speaker_wav=reference_audio, language='en', file_path=clone_path)

    return standard_path, clone_path

In [ ]:
demo = gr.Interface(
    fn=clone_voices,
    inputs=[
        gr.Textbox(label='Text to speak', placeholder='Type a sentence or two...', lines=3),
        gr.Audio(label='Voice sample to clone (record or upload, 6-10 seconds)',
                 sources=['microphone', 'upload'], type='filepath'),
    ],
    outputs=[
        gr.Audio(label='Standard AI voice'),
        gr.Audio(label='Cloned voice'),
    ],
    title='Voice Cloner',
    description=('ICS 499 capstone - Group 12 (Mubarek Idris & Mikita Misiulia). '
                 'Type text, record or upload a short voice sample, and compare a standard '
                 'AI voice with a clone of the sample voice.'),
    article=('**Responsible use:** only clone a voice with that person\'s permission. '
             'Cloning someone without consent can be used for fraud and impersonation. '
             'This demo is for coursework and keeps everything in your own Colab session.'),
)

# share=True prints a public gradio.live link - open THAT link to use/demo the app.
demo.launch(share=True)